# Part A Q4: Sentence Probabilities

In [7]:
from collections import Counter
from fractions import Fraction
from math import isclose, prod
from pathlib import Path

from nltk.lm import Laplace, MLE
from nltk.lm.vocabulary import Vocabulary
from nltk.util import everygrams


START_TOKEN = "<s>"
UNKNOWN_TOKEN = "<unk>"


# Walk up from the working directory to find Part_A/data, so the notebook runs
# regardless of whether the kernel starts in this folder, in Part_A, or at the repo root.
def find_data_file(filename):
    marker = Path("data") / filename
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).exists():
            return candidate / marker
        if (candidate / "Part_A" / marker).exists():
            return candidate / "Part_A" / marker
    raise FileNotFoundError(f"Could not locate Part_A/data/{filename} from {Path.cwd()}")


DATA_FILE = find_data_file("Data_3.txt")
print("Loaded Data_3.txt successfully.")

Loaded Data_3.txt successfully.


In [8]:
def load_sentences(file_path):
    # Extract only the lines that are actual padded sentences.
    with open(file_path, "r", encoding="utf-8") as file:
        sentence_lines = [
            line.strip()
            for line in file
            if line.strip().startswith(START_TOKEN)
        ]

    # The first three padded sentences are the training corpus.
    training_sentences = [line.split() for line in sentence_lines[:-1]]

    # The final padded sentence is the sentence whose probability is calculated.
    test_sentence = sentence_lines[-1].split()
    return training_sentences, test_sentence


def get_bigrams(sentence):
    # Convert a sentence into adjacent word pairs such as (<s>, I), (I, read).
    return list(zip(sentence, sentence[1:]))


def format_fraction(value):
    # Keep exact fraction output for manual probability reporting.
    return f"{value.numerator}/{value.denominator}"

## Corpus Counts and Vocabulary

In [9]:
training_sentences, test_sentence = load_sentences(DATA_FILE)

# Count individual words and word pairs from the training corpus.
unigram_counts = Counter()
bigram_counts = Counter()

for sentence in training_sentences:
    unigram_counts.update(sentence)
    bigram_counts.update(get_bigrams(sentence))

# Include every corpus token, including <s> and </s>, then add <unk>.
# This matches the vocabulary size used by the NLTK model.
vocabulary = sorted(set(unigram_counts) | {UNKNOWN_TOKEN})
vocabulary_size = len(vocabulary)
test_bigrams = get_bigrams(test_sentence)


print("=== Training Sentences ===")
for sentence in training_sentences:
    print(" ".join(sentence))

print("\n=== Test Sentence ===")
print(" ".join(test_sentence))

print("\n=== Vocabulary ===")
print(vocabulary)
print(f"Vocabulary size (including <s>, </s>, and <unk>): {vocabulary_size}")

=== Training Sentences ===
<s> He read a book </s>
<s> I read a different book </s>
<s> He read a book by Danielle </s>

=== Test Sentence ===
<s> I read a book by Danielle </s>

=== Vocabulary ===
['</s>', '<s>', '<unk>', 'Danielle', 'He', 'I', 'a', 'book', 'by', 'different', 'read']
Vocabulary size (including <s>, </s>, and <unk>): 11


## Unsmoothed Bigram Probability

In [10]:
print("\n=== Unsmoothed Bigram Model ===")
unsmoothed_probability = Fraction(1, 1)

# Unsmoothed formula:
# P(wi | wi-1) = Count(wi-1, wi) / Count(wi-1)
for previous_token, current_token in test_bigrams:
    bigram_count = bigram_counts[(previous_token, current_token)]
    previous_count = unigram_counts[previous_token]
    conditional_probability = Fraction(bigram_count, previous_count)
    unsmoothed_probability *= conditional_probability

    print(
        f"P({current_token} | {previous_token}) = "
        f"{bigram_count}/{previous_count} = {conditional_probability}"
    )

print(
    "Unsmoothed sentence probability = "
    f"{format_fraction(unsmoothed_probability)} = {float(unsmoothed_probability):.8f}"
)


=== Unsmoothed Bigram Model ===
P(I | <s>) = 1/3 = 1/3
P(read | I) = 1/1 = 1
P(a | read) = 3/3 = 1
P(book | a) = 2/3 = 2/3
P(by | book) = 1/3 = 1/3
P(Danielle | by) = 1/1 = 1
P(</s> | Danielle) = 1/1 = 1
Unsmoothed sentence probability = 2/27 = 0.07407407


## Laplace-Smoothed Bigram Probability

In [11]:
print("\n=== Smoothed Bigram Model (Add-One / Laplace) ===")
smoothed_probability = Fraction(1, 1)

# Add-one smoothing formula:
# P(wi | wi-1) = (Count(wi-1, wi) + 1) / (Count(wi-1) + V)
for previous_token, current_token in test_bigrams:
    bigram_count = bigram_counts[(previous_token, current_token)]
    previous_count = unigram_counts[previous_token]
    numerator = bigram_count + 1
    denominator = previous_count + vocabulary_size
    conditional_probability = Fraction(numerator, denominator)
    smoothed_probability *= conditional_probability

    print(
        f"P({current_token} | {previous_token}) = "
        f"({bigram_count}+1)/({previous_count}+{vocabulary_size}) = "
        f"{numerator}/{denominator} = {conditional_probability}"
    )

print(
    "Smoothed sentence probability = "
    f"{format_fraction(smoothed_probability)} = {float(smoothed_probability):.8f}"
)


=== Smoothed Bigram Model (Add-One / Laplace) ===
P(I | <s>) = (1+1)/(3+11) = 2/14 = 1/7
P(read | I) = (1+1)/(1+11) = 2/12 = 1/6
P(a | read) = (3+1)/(3+11) = 4/14 = 2/7
P(book | a) = (2+1)/(3+11) = 3/14 = 3/14
P(by | book) = (1+1)/(3+11) = 2/14 = 1/7
P(Danielle | by) = (1+1)/(1+11) = 2/12 = 1/6
P(</s> | Danielle) = (1+1)/(1+11) = 2/12 = 1/6
Smoothed sentence probability = 1/172872 = 0.00000578
